# Resolution GUI example

Crop a resolution (RRT) image, click two chart corners in a **popup window**,
then this notebook measures contrast. Plots stay **inside the notebook**.

**How to run**

1. Run the cells from top to bottom.
2. In *Choose your image*, either keep the example image or uncomment `IMAGE_PATH`.
3. When the keypoint window opens, click two corners, then close that window.
4. The remaining cells show plots here. If saving is on, files go in a results folder.


## 1. Setup


In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from qal import RrtAnalyzer, RrtDataPlotter, RrtROI
from qal.data import res_sample_1
from qal.util import load_grayscale_image


## 2. Choose your image

Leave `IMAGE_PATH` as `None` to use QAL's built-in example.

To use your own file, uncomment the `IMAGE_PATH` line and paste a path to a
TIFF, BMP, PNG, or JPEG.


In [ ]:
# Uncomment the next line and put the path to your image:
# IMAGE_PATH = "/path/to/your/image.tiff"

# Leave this as None to use the built-in example image.
IMAGE_PATH = None


## 3. Saving plots and data

When `SAVE_RESULTS` is `True`:

- If you set `IMAGE_PATH`, files go next to that image in `<image_name>_results/`
- If you use the example image, files go next to this notebook in `res_sample_1_results/`


In [ ]:
# Set this to True to write plots and CSV files.
SAVE_RESULTS = True

# File names inside the results folder (used only when SAVE_RESULTS is True):
SAVE_LINE_PROFILE_PLOT = "line_profiles.png"
SAVE_CONTRAST_PLOT = "percentage_contrast.png"
SAVE_CONTRAST_CSV = "percentage_contrast.csv"

# Contrast cutoff used to report the smallest resolved feature (percent).
RESOLUTION_THRESHOLD = 26


## 4. Load the image


In [ ]:
if IMAGE_PATH is None:
    image = res_sample_1()
    results_dir = Path("res_sample_1_results")
else:
    image = load_grayscale_image(IMAGE_PATH)
    image_file = Path(IMAGE_PATH).expanduser()
    results_dir = image_file.parent / f"{image_file.stem}_results"

if SAVE_RESULTS:
    results_dir.mkdir(parents=True, exist_ok=True)
    print(f"Results folder: {results_dir.resolve()}")
else:
    results_dir = None

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(image, cmap="gray")
ax.set_title("Loaded image")
ax.axis("off")
plt.show()


## 5. Crop the resolution chart


In [ ]:
rrt_roi = RrtROI()
cropped_image = rrt_roi.get_resolution_target_cropped(image, show_kp=False)
if cropped_image is None:
    raise RuntimeError("Could not find the resolution chart in this image.")

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(cropped_image, cmap="gray")
ax.set_title("Cropped image")
ax.axis("off")
plt.show()


## 6. Click two corners (popup window)

This cell opens a **separate window**. Click:

1. Top-left of the top bar of Group 0, Element 2
2. Bottom-right of the bottom bar of Group 0, Element 1

Then close the window to continue.


In [ ]:
# Use a desktop window for clicking, not an inline plot.
for backend in ("osx", "qt", "tk"):
    try:
        get_ipython().run_line_magic("matplotlib", backend)
        break
    except Exception:
        pass

rrt_roi.select_points_standard(cropped_image)


## 7. Put plots back in the notebook


In [ ]:
%matplotlib inline
plt.close("all")


## 8. Measure contrast


In [ ]:
if not rrt_roi.group_coordinates:
    raise RuntimeError("No corners yet. Finish the popup, then rerun that cell.")

analyzer = RrtAnalyzer()
percentage_contrast_df = analyzer.load_and_process_groups(
    cropped_image, rrt_roi.group_coordinates
)

above = percentage_contrast_df[
    percentage_contrast_df["Percentage Contrast"] >= RESOLUTION_THRESHOLD
]
if above.empty:
    print(f"All points are below {RESOLUTION_THRESHOLD}% contrast.")
else:
    last = above.iloc[-1]
    print(
        f"Smallest feature above {RESOLUTION_THRESHOLD}% contrast: "
        f"Group {int(last['Group'])}, Element {int(last['Element'])}, "
        f"{last['Resolution (lp/mm)']:.2f} lp/mm"
    )

percentage_contrast_df


## 9. Plot line profiles


In [ ]:
visualizer = RrtDataPlotter()
%matplotlib inline

visualizer.plot_line_profiles(
    cropped_image, rrt_roi.group_coordinates, percentage_contrast_df
)
if results_dir is not None:
    line_profile_path = results_dir / SAVE_LINE_PROFILE_PLOT
    plt.savefig(line_profile_path, bbox_inches="tight")
    print(f"Saved {line_profile_path}")


## 10. Plot contrast vs resolution


In [ ]:
visualizer.plot_percentage_contrast(
    percentage_contrast_df, resolution_threshold=RESOLUTION_THRESHOLD
)
if results_dir is not None:
    contrast_plot_path = results_dir / SAVE_CONTRAST_PLOT
    plt.savefig(contrast_plot_path, bbox_inches="tight")
    print(f"Saved {contrast_plot_path}")


## 11. Save the contrast table as CSV


In [ ]:
if results_dir is None:
    print("SAVE_RESULTS is False, so no CSV file was written.")
else:
    contrast_csv_path = results_dir / SAVE_CONTRAST_CSV
    percentage_contrast_df.to_csv(contrast_csv_path, index=False)
    print(f"Saved {contrast_csv_path}")
